# Heath–Jarrow–Morton (HJM) Term Structure Modeling of U.S. Interest Rates
Author: Maximilian Yap, Cornell University

<h1>Table of Contents<span class="tocSkip"></span></h1>



## 0. Front Matter & Reproducibility

**Last Updated:**  
12-Jan-2025

**Repository:**  
<[GitHub repository link](https://github.com/mmy32/HJM-Libor-Model)>  
**Commit hash:** <short hash>

**Environment Specification:**  
- Python: 3.9+  
- Core libraries: `numpy`, `pandas`, `scipy`, `scikit-learn`, `plotly`, `statsmodels`  
- Optional: `dvc` for data versioning

**Execution Contract:**  
This notebook is designed to be executed **top-to-bottom** without manual intervention.  
All parameters controlling calibration windows, tenors, factor counts, and simulation horizons are defined explicitly in Section 4.

### What This Notebook Produces
- A clean, reproducible pipeline from raw yield data to:
  - factor-extracted curve dynamics,
  - arbitrage-consistent HJM drift and volatility terms,
  - simulated future yield curve scenarios.

**Expected Runtime:**  
~30 minutes on a standard laptop for calibration and diagnostics.


## 1. Executive Summary

### Objective
The goal of this project is to **model, calibrate, and simulate the evolution of the interest rate term structure** using a Heath–Jarrow–Morton (HJM) framework calibrated to historical U.S. Treasury yield data.



### Modeling Assumptions (High-Level)
- Interest rate dynamics are adequately captured by a **low-dimensional factor structure**.
- Historical yield movements are informative about future volatility (stationarity assumption).
- No-arbitrage conditions are enforced through the HJM drift restriction.


## 2. Introduction: Yield Curve Modeling and the HJM Framework

### 2.1 The Problem of Yield Curve Fitting
Interest rates are observed at a discrete set of maturities, yet many financial applications, such as risk management, scenario analysis, and pricing, require a **continuous, arbitrage-consistent representation of the entire yield curve** as it evolves over time. Empirically, yield curves must satisfy several competing requirements: they should fit observed market data closely, evolve smoothly across maturities, and generate realistic dynamics over time. Naïve fitting approaches often fail one or more of these criteria, leading to unstable extrapolations, implausible curve shapes, or violations of no-arbitrage conditions.


The Heath–Jarrow–Morton (HJM) framework provides a theoretically rigorous solution by modeling the **entire forward rate curve as a stochastic process**. Rather than specifying dynamics for a single short rate, HJM directly characterizes the evolution of forward rates across maturities. Crucially, once the volatility structure of the forward curve is specified, the drift is uniquely determined by a no-arbitrage condition. This makes HJM a natural framework for generating arbitrage-free yield curve dynamics and coherent multi-maturity scenarios.

### 2.2 Limitations of a Naïve HJM Implementation
Despite its theoretical appeal, a basic HJM implementation faces several practical challenges:
- The forward curve is infinite-dimensional, making unrestricted volatility specification infeasible.
- Arbitrary volatility choices can lead to unstable or economically implausible dynamics.
- Estimating a high-dimensional volatility structure directly from data is noisy and prone to overfitting.
- Without dimensionality reduction, simulations become computationally expensive and difficult to interpret.

These issues limit the usefulness of HJM unless additional structure is imposed.

### 2.3 PCA as a Practical Resolution

Empirically, yield curve movements are highly correlated and can be well-approximated by a **small number of common factors**. Principal Component Analysis (PCA) exploits this structure by identifying the dominant modes of variation—commonly interpreted as level, slope, and curvature effects. By projecting historical yield (or forward rate) changes onto a low-dimensional factor space, PCA provides:
- a parsimonious and data-driven volatility specification,
- noise reduction and improved stability,
- interpretable economic dynamics.

Embedding these PCA-derived factors into the HJM framework yields a **low-dimensional, arbitrage-consistent model** that remains faithful to observed yield curve behavior.

### 2.5 Process Overview and Intended Usefulness
This notebook implements the following pipeline:
1. ingest and clean historical yield curve data,
2. extract dominant factors via PCA,
3. map factor volatilities into an HJM-consistent volatility structure,
4. compute the implied no-arbitrage drift,
5. simulate future yield curve scenarios and validate their properties.

The ultimate objective is to produce a **transparent, reproducible, and interpretable yield curve model** that balances theoretical soundness with empirical realism. While not intended as a production pricing system, the framework is designed to be a useful foundation for scenario generation, stress testing, and further extensions in quantitative fixed-income research.



## 3. Data & Provenance

The data source of this project is a panel of U.S. Treasury constant maturity yields obtained directly from the Federal Reserve Economic Data (FRED) database. 

The download script under `src/data_processing/data_download.py` queries a set of standard Treasury yield series identified by their FRED symbols (e.g., DGS1MO, DGS2, DGS10), each corresponding to a fixed maturity. These series are mapped to numeric tenors measured in years, allowing the yield curve to be represented on a common and interpretable maturity axis. Data are retrieved from a user-specified start date through the present, merged into a single time-indexed matrix, and aligned by date. The resulting structure is a rectangular panel with observation dates along the rows and maturities along the columns.

Raw yields from FRED are reported in percentage terms. The script converts these values into decimals to ensure internal consistency with continuous-time modeling conventions. Missing observations are forward-filled to maintain temporal continuity, after which any remaining incomplete rows are removed. This produces a dataset without gaps, suitable for principal component analysis and subsequent estimation of HJM volatility structures. While forward-filling imposes a strong assumption—namely that unobserved quotes remain unchanged—it avoids introducing artificial noise through interpolation and preserves the joint cross-sectional structure of the curve. The consequences of this choice are revisited later through diagnostic checks on factor stability and residual behavior.

Once constructed, the dataset is written to `data/treasury_yields.csv`. This file serves as the canonical input for the remainder of the project, including visualization, factor extraction, and simulation. All subsequent code assumes that this file exists and conforms to the schema produced by the download script.

The cell below executes the same logic as the standalone script, generating the yield matrix within the notebook environment and returning it as a pandas DataFrame. Successful execution confirms that the data required for the HJM calibration pipeline have been built correctly and are available for further analysis.



In [1]:
import sys
from pathlib import Path

# If your notebook is in ./notebooks, this points to repo root
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src").exists() is False:
    PROJECT_ROOT = Path.cwd().parent  # adjust if needed

sys.path.insert(0, str(PROJECT_ROOT))

In [2]:

from src.data_processing.data_download import get_treasury_matrix

df = get_treasury_matrix(start_date="2018-01-01")
df.head()

--- Starting Data Ingestion ---
Targeting 11 tenors from 2018-01-01 to 2026-02-05
[*] Fetching DGS1MO (0.0833 year tenor)... Done.
[*] Fetching DGS3MO (0.25 year tenor)... Done.
[*] Fetching DGS6MO (0.5 year tenor)... Done.
[*] Fetching DGS1 (1.0 year tenor)... Done.
[*] Fetching DGS2 (2.0 year tenor)... Done.
[*] Fetching DGS3 (3.0 year tenor)... Done.
[*] Fetching DGS5 (5.0 year tenor)... Done.
[*] Fetching DGS7 (7.0 year tenor)... Done.
[*] Fetching DGS10 (10.0 year tenor)... Done.
[*] Fetching DGS20 (20.0 year tenor)... Done.
[*] Fetching DGS30 (30.0 year tenor)... Done.
--- Processing Data ---
Cleaned data: Kept 2021 of 2112 rows after removing NaNs.
[SUCCESS] Data saved to data/treasury_yields.csv


,0.0833,0.2500,0.5000,1.0000,2.0000,3.0000,5.0000,7.0000,10.0000,20.0000,30.0000
DATE,,,,,,,,,,,
2018-01-02,0.0129,0.0144,0.0161,0.0183,0.0192,0.0201,0.0225,0.0238,0.0246,0.0264,0.0281
2018-01-03,0.0129,0.0141,0.0159,0.0181,0.0194,0.0202,0.0225,0.0237,0.0244,0.0262,0.0278
2018-01-04,0.0128,0.0141,0.0160,0.0182,0.0196,0.0205,0.0227,0.0238,0.0246,0.0262,0.0279
2018-01-05,0.0127,0.0139,0.0158,0.0180,0.0196,0.0206,0.0229,0.0240,0.0247,0.0264,0.0281
2018-01-08,0.0130,0.0145,0.0160,0.0179,0.0196,0.0207,0.0229,0.0241,0.0249,0.0265,0.0281


After execution, the DataFrame index consists of observation dates, the columns correspond to maturities expressed as floating-point numbers in years, and the entries contain Treasury yields in decimal form. This representation provides a direct and transparent link between the raw market data and the stochastic term-structure model developed in the subsequent sections.
